# E4 — Statistical Power Analysis (the Gate 1 deliverable)

**Experiment ID:** `E4`. **Specification:** `EXPERIMENT_PLAN.md` §E4. **Governing rules:** `CLAUDE.md`.

**The question this answers.** *Given how few high-risk events exist, can empirical coverage be
estimated with useful precision — marginally, and per mission group?* This is the project's primary
go/no-go checkpoint (`PROJECT_KNOWLEDGE.md` §14 R1; Assumption 5).

**Pre-registration.** The nominal level (90%), the "useful precision" bar (95% CI half-width
≤ 5 percentage points), and the merging-threshold rule (Q-CONF-02 option (b)) were written to
`DECISIONS.md` — marked *PROPOSED — awaiting Sidh's confirmation* — and encoded in
`config/default.yaml` **before** this simulation was run (`CLAUDE.md` §3).

**Scope limit.** This notebook produces the Gate 1 decision table. It does **not** issue the
GO / PIVOT / NO-GO call, and it does not finalise the merge threshold — both are Sidh's, exactly as
the Assumption-A4 call was in Phase 0 (`CLAUDE.md` §3, §11).

**Inputs.** Counts and labels only (`EXPERIMENT_PLAN.md` E4). No model is trained, fitted, or used
anywhere in this notebook; E4 is a Monte Carlo study of attainable precision.

## 0. The simulation model

Two independent sources of randomness limit how precisely coverage can be pinned down. Both are
simulated — conflating them, or omitting the first, would understate the uncertainty.

**1. Calibration-set randomness.** For split conformal with $n_{cal}$ calibration points at nominal
level $1-\alpha$, the coverage achieved on a fresh test point, *conditional on the calibration
draw*, is itself random:

$$C \mid \text{calibration} \;\sim\; \mathrm{Beta}\!\left(n_{cal}+1-\ell,\; \ell\right),
\qquad \ell = \lfloor \alpha (n_{cal}+1) \rfloor$$

(the standard finite-sample split-conformal coverage distribution). Its mean is $\ge 1-\alpha$ and
its spread shrinks with $n_{cal}$ — this is exactly what varying the calibration fraction changes.

**2. Test-set estimation noise.** Given a conditional coverage $p$, each of the $n_{test}$
evaluation events contributes a Bernoulli$(p)$ indicator, so the estimate is
$\mathrm{Binomial}(n_{test}, p)/n_{test}$.

For each configuration we repeat (1) then (2) and compute a 95% CI on the coverage estimate two
ways: the **event-level bootstrap percentile CI** that `EXPERIMENT_PLAN.md` E4 specifies, and the
**exact Clopper–Pearson interval**. Reported precision is the **median 95% CI half-width** across
repetitions, in percentage points.

Both are reported because the bootstrap has a failure mode at exactly the group sizes this
experiment cares about — see §3a, which is a finding of this experiment, not a footnote.
Clopper–Pearson is the **primary** measure; `OPEN_QUESTIONS.md` Q-STAT-03 already recommends
reporting both (option (c)), so this is the documented intent rather than a substitution.

The binding constraint in this project is $n_{test}$ = the number of **high-risk** events available
for evaluation, which E1 established is 150 marginally and far smaller per mission.

In [ ]:
# --- Setup, configuration, and provenance stamp (invariant I4) -------------------------------
import json, subprocess, sys
from datetime import datetime, timezone

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from kelvins_conformal.config import REPO_ROOT, load_config
from kelvins_conformal import data as kcdata
from kelvins_conformal import power as kcpower

cfg = load_config()
FIGDIR = cfg.path("figures_dir"); FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR = cfg.path("tables_dir"); TABDIR.mkdir(parents=True, exist_ok=True)

def git_sha() -> str:
    try:
        out = subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(REPO_ROOT),
                             capture_output=True, text=True, check=True)
        return out.stdout.strip()
    except Exception:
        return "UNAVAILABLE (working tree is not a git repository)"

P = cfg.power
NOMINAL = P.nominal_coverage_primary
BAR = P.useful_half_width_pp
BAR2 = P.secondary_half_width_pp

PROVENANCE = {
    "experiment_ids": ["E4"],
    "git_commit_sha": git_sha(),
    "config_hash": cfg.config_hash,
    "seed": cfg.seed,
    "nominal_coverage_primary": NOMINAL,
    "useful_half_width_pp": BAR,
    "calibration_fractions": list(P.calibration_fractions),
    "n_simulations": P.n_simulations,
    "bootstrap_resamples": cfg.bootstrap.n_resamples,
    "bootstrap_unit": cfg.bootstrap.unit,
    "preregistration_status": "PROPOSED - awaiting Sidh's confirmation (DECISIONS.md)",
    "executed_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version.split()[0],
}
print(json.dumps(PROVENANCE, indent=2))

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIGDIR / f"{name}.{ext}", dpi=160, bbox_inches="tight")
    print(f"saved: reports/figures/{name}.png|pdf")

def save_table(df, name):
    df.to_csv(TABDIR / f"{name}.csv", index=True)
    print(f"saved: reports/tables/{name}.csv")

CTRAIN, CTEST = "#0072B2", "#D55E00"

In [ ]:
# --- Pre-registered settings actually in force ------------------------------------------------
print(f"nominal coverage (primary)   : {NOMINAL:.0%}")
print(f"nominal coverage (secondary) : {[f'{x:.0%}' for x in P.nominal_coverage_secondary]}")
print(f"calibration fractions        : {[f'{f:.0%}' for f in P.calibration_fractions]}")
print(f"useful-precision bar         : 95% CI half-width <= {BAR:g} pp")
print(f"secondary (looser) bar       : <= {BAR2:g} pp")
print(f"Monte Carlo repetitions      : {P.n_simulations:,}")
print(f"bootstrap resamples / rep    : {cfg.bootstrap.n_resamples:,} (event-level)")
print(f"seed                         : {cfg.seed}")
print("\nSTATUS: PROPOSED - not yet confirmed by Sidh. Any 'meets the bar' language below is")
print("conditional on this bar being accepted as written.")

## 1. The counts that constrain everything

Calibration sizes come from the training pool; evaluation counts come from the official test set's
high-risk events. Both are read from the data, not assumed.

In [ ]:
# --- Establish the real counts ----------------------------------------------------------------
events = kcdata.load_events(cfg)
per_event = kcdata.event_level_frame(events)
train_ev = per_event[per_event["split"] == "train"]
test_ev = per_event[per_event["split"] == "test"]

n_train_events = len(train_ev)
n_train_hr = int(train_ev["is_high_risk"].sum())
n_test_events = len(test_ev)
n_test_hr = int(test_ev["is_high_risk"].sum())

print(f"train events              : {n_train_events:,}   (high-risk: {n_train_hr:,})")
print(f"official test events      : {n_test_events:,}   (high-risk: {n_test_hr:,})")

cal_sizes = {f: int(round(f * n_train_events)) for f in P.calibration_fractions}
cal_sizes_hr = {f: int(round(f * n_train_hr)) for f in P.calibration_fractions}
sizes_tbl = pd.DataFrame({
    "calibration fraction": [f"{f:.0%}" for f in P.calibration_fractions],
    "n_cal (all train events)": list(cal_sizes.values()),
    "n_cal (high-risk train only)": list(cal_sizes_hr.values()),
}).set_index("calibration fraction")
display(sizes_tbl)
save_table(sizes_tbl, "e4_calibration_sizes")
print("\nBoth calibration regimes are carried through: marginal calibration uses the whole train")
print("pool; a high-risk-stratified calibration (should Phase 3's Q-CONF-01 design choose it)")
print("collapses n_cal by ~36x, which materially changes the picture — reported in section 4.")

In [ ]:
# --- Per-mission high-risk counts on the official test set -------------------------------------
mission_tbl = (per_event
    .pivot_table(index="mission_id", columns="split", values="event_uid",
                 aggfunc="count", fill_value=0)
    .join(per_event[per_event["is_high_risk"]]
          .pivot_table(index="mission_id", columns="split", values="event_uid",
                       aggfunc="count", fill_value=0)
          .add_prefix("hr_"), how="left")
    .fillna(0).astype(int))
mission_tbl = mission_tbl.rename(columns={"train": "events_train", "test": "events_test",
                                          "hr_train": "hr_train", "hr_test": "hr_test"})
shared = mission_tbl[(mission_tbl["events_train"] > 0) & (mission_tbl["events_test"] > 0)]
shared = shared.sort_values("hr_test", ascending=False)

print(f"missions present in BOTH splits: {len(shared)}")
print(f"total high-risk test events across them: {int(shared['hr_test'].sum())}")
print(f"largest single-mission high-risk test count: {int(shared['hr_test'].max())} "
      f"(mission {int(shared['hr_test'].idxmax())})")
display(shared)
save_table(shared, "e4_per_mission_counts")

## 2. Marginal precision — can coverage be estimated at all?

In [ ]:
# --- Marginal power sweep ----------------------------------------------------------------------
marginal_rows = []
for frac, n_cal in cal_sizes.items():
    for level in (NOMINAL, *P.nominal_coverage_secondary):
        res = kcpower.simulate_precision(
            n_cal, n_test_hr, nominal=level,
            n_simulations=P.n_simulations, n_bootstrap=cfg.bootstrap.n_resamples,
            level=0.95, seed=cfg.seed,
        )
        marginal_rows.append({
            "calibration fraction": f"{frac:.0%}", "n_cal": n_cal,
            "nominal": f"{level:.0%}", "n_test (high-risk)": n_test_hr,
            "median CP half-width (pp)": res.median_cp_half_width_pp,
            "median bootstrap half-width (pp)": res.median_half_width_pp,
            "bootstrap degenerate frac": res.degenerate_fraction,
            "5th pct (pp)": res.q05_half_width_pp,
            "95th pct (pp)": res.q95_half_width_pp,
            "sd of estimate (pp)": res.sd_coverage_estimate_pp,
            "calibration term (pp)": res.beta_sd_pp,
            "test-set term (pp)": res.binomial_sd_pp,
            f"meets {BAR:g}pp bar": res.meets(BAR),
            f"meets {BAR2:g}pp bar": res.meets(BAR2),
        })

marginal = pd.DataFrame(marginal_rows)
primary_marginal = marginal[marginal["nominal"] == f"{NOMINAL:.0%}"].set_index("calibration fraction")
display(primary_marginal.round(3))
save_table(marginal.set_index(["calibration fraction", "nominal"]), "e4_marginal_precision")

print(f"\nAt the primary {NOMINAL:.0%} level with all {n_test_hr} high-risk test events:")
for _, r in primary_marginal.iterrows():
    flag = "MEETS" if r[f"meets {BAR:g}pp bar"] else "does NOT meet"
    print(f"  n_cal={r['n_cal']:>5,}  ->  CP half-width {r['median CP half-width (pp)']:.2f} pp"
          f"   ({flag} the {BAR:g} pp bar)")

In [ ]:
# --- Which term dominates? --------------------------------------------------------------------
dom = primary_marginal[["calibration term (pp)", "test-set term (pp)"]].copy()
dom["ratio test/calibration"] = dom["test-set term (pp)"] / dom["calibration term (pp)"]
display(dom.round(3))
print("The test-set term dominates at every calibration fraction: with only "
      f"{n_test_hr} high-risk\ntest events, precision is limited by the EVALUATION set, not by how "
      "much data is\nspent on calibration. Increasing the calibration fraction therefore buys very "
      "little\nprecision — a directly actionable Gate-1 observation.")

## 3. Per-mission precision — the group-conditional question

In [ ]:
# --- Per-mission power sweep at the primary level ----------------------------------------------
group_rows = []
for frac, n_cal in cal_sizes.items():
    for mission, row in shared.iterrows():
        n_hr = int(row["hr_test"])
        if n_hr < 1:
            group_rows.append({
                "calibration fraction": f"{frac:.0%}", "mission_id": int(mission),
                "hr_test": n_hr, "median CP half-width (pp)": np.nan,
                "median bootstrap half-width (pp)": np.nan,
                f"meets {BAR:g}pp bar": False, f"meets {BAR2:g}pp bar": False,
                "status": "UNSCORABLE (no high-risk test events)",
            })
            continue
        res = kcpower.simulate_precision(
            n_cal, n_hr, nominal=NOMINAL,
            n_simulations=max(200, P.n_simulations // 5),
            n_bootstrap=max(500, cfg.bootstrap.n_resamples // 4),
            level=0.95, seed=cfg.seed,
        )
        group_rows.append({
            "calibration fraction": f"{frac:.0%}", "mission_id": int(mission),
            "hr_test": n_hr,
            "median CP half-width (pp)": res.median_cp_half_width_pp,
            "median bootstrap half-width (pp)": res.median_half_width_pp,
            "bootstrap degenerate frac": res.degenerate_fraction,
            f"meets {BAR:g}pp bar": res.meets(BAR),
            f"meets {BAR2:g}pp bar": res.meets(BAR2),
            "status": "estimable" if res.meets(BAR2) else "TOO IMPRECISE",
        })

groups = pd.DataFrame(group_rows)
save_table(groups.set_index(["calibration fraction", "mission_id"]), "e4_per_mission_precision")

# Present at the primary calibration fraction (the middle one) for readability.
mid_frac = f"{P.calibration_fractions[len(P.calibration_fractions)//2]:.0%}"
gview = groups[groups["calibration fraction"] == mid_frac].set_index("mission_id")
gview = gview.sort_values("hr_test", ascending=False)
display(gview.round(2))

n_meet = int(gview[f"meets {BAR:g}pp bar"].sum())
n_meet2 = int(gview[f"meets {BAR2:g}pp bar"].sum())
print(f"\nAt calibration fraction {mid_frac}, nominal {NOMINAL:.0%}:")
print(f"  missions meeting the {BAR:g} pp bar : {n_meet} of {len(gview)}")
print(f"  missions meeting the {BAR2:g} pp bar: {n_meet2} of {len(gview)}")

In [ ]:
# --- Derive the Q-CONF-02 merging floor from the power curve ------------------------------------
n_cal_mid = cal_sizes[P.calibration_fractions[len(P.calibration_fractions)//2]]

sweep_rows = []
for n in P.group_size_grid:
    res = kcpower.simulate_precision(
        n_cal_mid, int(n), nominal=NOMINAL,
        n_simulations=max(200, P.n_simulations // 5),
        n_bootstrap=max(500, cfg.bootstrap.n_resamples // 4),
        level=0.95, seed=cfg.seed,
    )
    sweep_rows.append({"group size (high-risk events)": int(n),
                       "median CP half-width (pp)": res.median_cp_half_width_pp,
                       "median bootstrap half-width (pp)": res.median_half_width_pp,
                       "bootstrap degenerate frac": res.degenerate_fraction,
                       f"meets {BAR:g}pp": res.meets(BAR),
                       f"meets {BAR2:g}pp": res.meets(BAR2)})
sweep = pd.DataFrame(sweep_rows).set_index("group size (high-risk events)")
display(sweep.round(2))
save_table(sweep, "e4_group_size_sweep")

floor_primary = kcpower.minimum_group_size(
    n_cal_mid, bar_pp=BAR, size_grid=P.group_size_grid, nominal=NOMINAL,
    n_simulations=max(200, P.n_simulations // 5),
    n_bootstrap=max(500, cfg.bootstrap.n_resamples // 4), seed=cfg.seed)
floor_secondary = kcpower.minimum_group_size(
    n_cal_mid, bar_pp=BAR2, size_grid=P.group_size_grid, nominal=NOMINAL,
    n_simulations=max(200, P.n_simulations // 5),
    n_bootstrap=max(500, cfg.bootstrap.n_resamples // 4), seed=cfg.seed)

print(f"\nDERIVED MERGING FLOOR (Q-CONF-02 option (b)):")
print(f"  at the {BAR:g} pp bar : {floor_primary} high-risk events per group")
print(f"  at the {BAR2:g} pp bar: {floor_secondary} high-risk events per group")
print(f"\nFor reference, OPEN_QUESTIONS.md Q-CONF-02 option (a) proposed a fixed floor of 30.")

In [ ]:
# --- Figure: CI half-width vs calibration size, marginal and per group -------------------------
fig = plt.figure(figsize=(13.5, 8.6))
gs = fig.add_gridspec(3, 5, hspace=0.55, wspace=0.35)

# Panel A (top-left, spanning 2 cols): marginal.
axA = fig.add_subplot(gs[0, :2])
for level, style in zip((NOMINAL, *P.nominal_coverage_secondary), ("-o", "--s", ":^")):
    sub = marginal[marginal["nominal"] == f"{level:.0%}"]
    axA.plot(sub["n_cal"], sub["median CP half-width (pp)"], style,
             label=f"nominal {level:.0%}")
axA.axhline(BAR, color="crimson", ls="--", lw=1.3, label=f"{BAR:g} pp bar")
axA.axhline(BAR2, color="grey", ls=":", lw=1.2, label=f"{BAR2:g} pp bar")
axA.set_xlabel("calibration set size (events)")
axA.set_ylabel("median 95% CI half-width (pp)")
axA.set_title(f"(A) Marginal — all {n_test_hr} high-risk test events", fontsize=10)
axA.legend(frameon=False, fontsize=7.5)

# Panel B (top-right, spanning 3 cols): half-width vs group size.
axB = fig.add_subplot(gs[0, 2:])
axB.plot(sweep.index, sweep["median CP half-width (pp)"], "-o", color=CTRAIN,
         label="Clopper-Pearson (primary)")
axB.plot(sweep.index, sweep["median bootstrap half-width (pp)"], "--x", color="grey", ms=4,
         label="bootstrap (degenerates at small n)")
axB.axhline(BAR, color="crimson", ls="--", lw=1.3, label=f"{BAR:g} pp bar")
axB.axhline(BAR2, color="grey", ls=":", lw=1.2, label=f"{BAR2:g} pp bar")
if floor_primary:
    axB.axvline(floor_primary, color="crimson", ls="-.", lw=1.1,
                label=f"derived floor = {floor_primary}")
axB.scatter(shared["hr_test"], [
    groups[(groups["calibration fraction"] == mid_frac) &
           (groups["mission_id"] == m)]["median CP half-width (pp)"].iloc[0]
    for m in shared.index], color=CTEST, zorder=5, s=28, label="actual missions")
axB.set_xscale("log"); axB.set_xlabel("high-risk events in group (log scale)")
axB.set_ylabel("median half-width (pp)")
axB.set_title("(B) Precision vs group size, with the real missions overlaid", fontsize=10)
axB.legend(frameon=False, fontsize=7.5)

# Panels C..: small multiples, one per mission.
missions = list(shared.index)
for i, mission in enumerate(missions[:10]):
    ax = fig.add_subplot(gs[1 + i // 5, i % 5])
    sub = groups[groups["mission_id"] == mission]
    ax.plot([cal_sizes[f] for f in P.calibration_fractions],
            sub["median CP half-width (pp)"], "-o", ms=3.5, color=CTRAIN)
    ax.axhline(BAR, color="crimson", ls="--", lw=1.0)
    ax.axhline(BAR2, color="grey", ls=":", lw=1.0)
    hr = int(shared.loc[mission, "hr_test"])
    ax.set_title(f"mission {int(mission)}  (n_HR={hr})", fontsize=8)
    ax.tick_params(labelsize=6)
    ax.set_ylim(0, max(60, float(sub["median CP half-width (pp)"].max()) * 1.15))
    if i % 5 == 0:
        ax.set_ylabel("half-width (pp)", fontsize=7)
    if i // 5 == 1:
        ax.set_xlabel("n_cal", fontsize=7)

fig.suptitle("E4  Attainable precision of a coverage estimate "
             f"(nominal {NOMINAL:.0%}, 95% CI half-width)", fontsize=12, y=0.97)
save_fig(fig, "e4_power_ci_halfwidth")
plt.show()

## 4. The high-risk-stratified calibration regime

If Phase 3's calibration unit (Q-CONF-01, still open) turns out to be *within the high-risk
stratum* rather than the full pool, the calibration set collapses from thousands of events to
tens. That is a materially different power picture and Gate 1 should see it.

In [ ]:
# --- Same sweep, but calibrating within the high-risk stratum only -----------------------------
strat_rows = []
for frac, n_cal_hr in cal_sizes_hr.items():
    try:
        res = kcpower.simulate_precision(
            n_cal_hr, n_test_hr, nominal=NOMINAL,
            n_simulations=max(200, P.n_simulations // 5),
            n_bootstrap=max(500, cfg.bootstrap.n_resamples // 4),
            level=0.95, seed=cfg.seed,
        )
        strat_rows.append({
            "calibration fraction": f"{frac:.0%}", "n_cal (high-risk only)": n_cal_hr,
            "median CI half-width (pp)": res.median_half_width_pp,
            "calibration term (pp)": res.beta_sd_pp,
            "test-set term (pp)": res.binomial_sd_pp,
            f"meets {BAR:g}pp bar": res.meets(BAR),
            "status": "ok",
        })
    except ValueError as e:
        strat_rows.append({
            "calibration fraction": f"{frac:.0%}", "n_cal (high-risk only)": n_cal_hr,
            "median CI half-width (pp)": np.nan,
            "calibration term (pp)": np.nan, "test-set term (pp)": np.nan,
            f"meets {BAR:g}pp bar": False,
            "status": f"INFEASIBLE: {e}",
        })

strat = pd.DataFrame(strat_rows).set_index("calibration fraction")
display(strat.round(3))
save_table(strat, "e4_high_risk_stratified_calibration")
print("If Phase 3 calibrates within the high-risk stratum, the calibration term stops being")
print("negligible and becomes a real contributor to imprecision. Q-CONF-01 is therefore not a")
print("purely cosmetic design choice — flagged for the Phase 3 design review, not decided here.")

## 3a. A methodological finding: the bootstrap degenerates on small groups

**This changes the merging floor materially, so it is reported before the decision table.**

A percentile bootstrap of a proportion has **exactly zero width** whenever the sample is all-ones
(or all-zeros): every resample then has the same mean. That is not an exotic corner case here. At
nominal 90% coverage, the probability that *every* event in a group is covered is $0.9^{n}$ — which
is **59% at n = 5** and still 12% at n = 20. The *median* bootstrap half-width across simulations
is therefore literally 0 for small groups, which would make a 5-event mission look infinitely
precise.

A first run of this notebook did exactly that, reporting a derived merging floor of 5 high-risk
events and identical group counts at the 5 pp and 10 pp bars — an impossible result that traced to
this degeneracy, not to the data. The pre-registered bar is unchanged; the *estimator* was wrong and
is fixed. Precision is now judged on Clopper–Pearson, which is exact and non-degenerate at the
boundary (for $k = n$ it returns $[lpha^{1/n}, 1]$, correctly wide). The bootstrap is still
reported alongside, together with the fraction of simulations in which it collapsed, so the failure
mode is visible rather than silently trusted.

`tests/test_power.py` pins this behaviour so it cannot silently return.

In [ ]:
# --- Quantify the degeneracy directly ---------------------------------------------------------
degen_rows = []
for n in (5, 10, 20, 32, 75, 150):
    res = kcpower.simulate_precision(
        n_cal_mid, n, nominal=NOMINAL, n_simulations=max(200, P.n_simulations // 5),
        n_bootstrap=max(500, cfg.bootstrap.n_resamples // 4), seed=cfg.seed)
    degen_rows.append({
        "group size (high-risk events)": n,
        "P(all covered) = 0.9^n": NOMINAL ** n,
        "bootstrap degenerate frac": res.degenerate_fraction,
        "median bootstrap half-width (pp)": res.median_half_width_pp,
        "median CP half-width (pp)": res.median_cp_half_width_pp,
    })
degen = pd.DataFrame(degen_rows).set_index("group size (high-risk events)")
display(degen.round(3))
save_table(degen, "e4_bootstrap_degeneracy")
print("Where the 'degenerate frac' exceeds 0.5 the median bootstrap half-width collapses to 0 and")
print("is meaningless. The CP column stays honest across the whole range — which is why the")
print("merging floor and every 'meets the bar' verdict below are judged on it.")

## 5. THE GATE 1 DECISION TABLE

Everything Gate 1 needs, in one place. **No decision is taken here.**

In [ ]:
# --- Gate 1 decision table ----------------------------------------------------------------------
gate_rows = []

# Marginal rows, one per calibration fraction.
for _, r in primary_marginal.iterrows():
    gate_rows.append({
        "scope": "MARGINAL (all high-risk test events)",
        "group": "-",
        "n_HR": n_test_hr,
        "n_cal": int(r["n_cal"]),
        "median CP half-width (pp)": round(float(r["median CP half-width (pp)"]), 2),
        f"meets {BAR:g}pp": bool(r[f"meets {BAR:g}pp bar"]),
        f"meets {BAR2:g}pp": bool(r[f"meets {BAR2:g}pp bar"]),
        "verdict": ("estimable at the pre-registered bar"
                    if r[f"meets {BAR:g}pp bar"] else
                    ("estimable only at the looser bar"
                     if r[f"meets {BAR2:g}pp bar"] else "NOT estimable")),
    })

# Per-mission rows at the middle calibration fraction.
for mission, r in gview.iterrows():
    n_hr = int(r["hr_test"])
    if n_hr == 0:
        verdict = "UNSCORABLE — no high-risk test events"
    elif r[f"meets {BAR:g}pp bar"]:
        verdict = "estimable at the pre-registered bar"
    elif r[f"meets {BAR2:g}pp bar"]:
        verdict = "estimable only at the looser bar"
    else:
        verdict = f"FLAGGED FOR MERGING (below the derived floor of {floor_primary})"
    gate_rows.append({
        "scope": f"PER MISSION (cal {mid_frac})",
        "group": f"mission {int(mission)}",
        "n_HR": n_hr,
        "n_cal": int(n_cal_mid),
        "median CP half-width (pp)": (round(float(r["median CP half-width (pp)"]), 2)
                                      if n_hr > 0 else np.nan),
        f"meets {BAR:g}pp": bool(r[f"meets {BAR:g}pp bar"]),
        f"meets {BAR2:g}pp": bool(r[f"meets {BAR2:g}pp bar"]),
        "verdict": verdict,
    })

gate1 = pd.DataFrame(gate_rows).set_index(["scope", "group"])
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 60)
display(gate1)
save_table(gate1, "e4_GATE1_decision_table")

In [ ]:
# --- Gate 1 summary block ------------------------------------------------------------------------
marg_ok_primary = bool(primary_marginal[f"meets {BAR:g}pp bar"].any())
marg_ok_secondary = bool(primary_marginal[f"meets {BAR2:g}pp bar"].any())
best_marg = float(primary_marginal["median CP half-width (pp)"].min())
groups_meeting_primary = int(gview[f"meets {BAR:g}pp bar"].sum())
groups_meeting_secondary = int(gview[f"meets {BAR2:g}pp bar"].sum())
largest_group = int(shared["hr_test"].max())

summary = pd.DataFrame([
    {"item": "Nominal level (pre-registered, PROPOSED)", "value": f"{NOMINAL:.0%}"},
    {"item": "Useful-precision bar (pre-registered, PROPOSED)", "value": f"{BAR:g} pp half-width"},
    {"item": "High-risk test events available (marginal)", "value": n_test_hr},
    {"item": "Best marginal half-width achieved", "value": f"{best_marg:.2f} pp"},
    {"item": "Marginal coverage estimable at the bar?", "value": "YES" if marg_ok_primary else "NO"},
    {"item": "Missions sharing both splits", "value": len(shared)},
    {"item": "Largest single-mission high-risk test count", "value": largest_group},
    {"item": f"Missions meeting the {BAR:g} pp bar", "value": f"{groups_meeting_primary} of {len(gview)}"},
    {"item": f"Missions meeting the {BAR2:g} pp bar", "value": f"{groups_meeting_secondary} of {len(gview)}"},
    {"item": "Derived merging floor (Q-CONF-02 opt. b)", "value": f"{floor_primary} high-risk events"},
    {"item": "EXPERIMENT_PLAN E4 minimum bar (marginal estimable)",
     "value": "MET" if marg_ok_primary else "NOT MET"},
    {"item": "EXPERIMENT_PLAN E4 stretch bar (2-3 groups estimable)",
     "value": "MET" if groups_meeting_primary >= 2 else "NOT MET"},
]).set_index("item")
display(summary)
save_table(summary, "e4_gate1_summary")

In [ ]:
# --- Closing statement ------------------------------------------------------------------------
print(f"""
E4 / GATE 1 — WHAT THE SIMULATION SHOWS (measurement only)

 MARGINAL
  * With all {n_test_hr} high-risk test events, the median 95% CI half-width on empirical
    coverage is {best_marg:.2f} pp at the best calibration fraction — {'AT OR INSIDE' if marg_ok_primary else 'OUTSIDE'}
    the pre-registered {BAR:g} pp bar.
  * Precision is limited by the EVALUATION set, not the calibration set: the test-set term
    dominates the calibration term at every fraction tried, so moving from 10% to 30%
    calibration buys almost nothing. EXPERIMENT_PLAN's E4 minimum bar is
    {'MET' if marg_ok_primary else 'NOT MET'}.

 PER GROUP
  * The largest single mission has only {largest_group} high-risk test events.
    {groups_meeting_primary} of {len(gview)} missions meet the {BAR:g} pp bar;
    {groups_meeting_secondary} of {len(gview)} meet the looser {BAR2:g} pp bar.
  * The power-derived merging floor (Q-CONF-02 option (b)) is {floor_primary} high-risk events
    per group. Every mission below it is FLAGGED FOR MERGING in the table above, not silently
    reported.
  * EXPERIMENT_PLAN's E4 stretch bar (group-conditional estimable for 2-3 groups) is
    {'MET' if groups_meeting_primary >= 2 else 'NOT MET'}.

 CAVEAT CARRIED FORWARD
  * If Phase 3's calibration unit (Q-CONF-01) is the high-risk stratum rather than the full
    training pool, n_cal collapses by roughly 36x and the calibration term stops being
    negligible. Section 4 quantifies that regime.

WHAT THIS NOTEBOOK DOES NOT DO (CLAUDE.md §3, §10, §11):
 * It does not make the GO / PIVOT / NO-GO call.
 * It does not finalise the group-merging threshold — {floor_primary} is DERIVED from a bar that
   is still marked PROPOSED in DECISIONS.md, and both are Sidh's to confirm or revise.
 * It does not freeze Q-STAT-01 (nominal levels) or Q-STAT-02 (success margin), though it
   supplies the numbers those decisions need.
""")

(cfg.path("reports_dir") / "01_power_analysis_provenance.json").write_text(
    json.dumps(PROVENANCE, indent=2), encoding="utf-8")
print("provenance sidecar: reports/01_power_analysis_provenance.json")